# HazardWatch — fire & smoke camera model (D-Fire → YOLO11n)

Trains the model that fills `YOLO_Flame_Detected` / `YOLO_Smoke_Detected`.

* **Runtime:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.
* **Data:** D-Fire (≈21,000 images, YOLO labels). **Class 0 = smoke, 1 = fire** (checked on the labels).
* **Saved to your Google Drive:** `MyDrive/HazardWatch/dfire/` (checkpoints every epoch, so a disconnect can resume).
* **Time:** about 2–3 hours for 40 epochs on a free T4. Keep this tab open.
* Ultralytics YOLO is AGPL-3.0 licensed; D-Fire: cite Venâncio et al., Neural Computing and Applications (2022).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."


In [ ]:
# 1. Google Drive (for checkpoints, results and the D-Fire backup copy). Approve the pop-up.
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/HazardWatch/dfire'
os.makedirs(OUT, exist_ok=True)
print('Results will be saved in', OUT)

In [ ]:
!pip -q install ultralytics

In [ ]:
# 2. Get D-Fire: (a) the authors' public OneDrive link, or (b) MyDrive/HazardWatch/D-Fire.zip if you uploaded it.
import base64, glob, os, subprocess, zipfile
DATA = '/content/dfire'
os.makedirs(DATA, exist_ok=True)
ZIP = '/content/D-Fire.zip'
DRIVE_ZIP = '/content/drive/MyDrive/HazardWatch/D-Fire.zip'
ONEDRIVE = 'https://1drv.ms/u/c/c0bd25b6b048b01d/EbLgD7bES4FDvUN37Grxn8QBF5gIBBc7YV2qklF08GCiBw'

def onedrive_direct(url):
    token = base64.urlsafe_b64encode(url.encode()).decode().rstrip('=')
    return f'https://api.onedrive.com/v1.0/shares/u!{token}/root/content'

if os.path.exists(DRIVE_ZIP):
    print('Copying D-Fire.zip from your Drive ...'); subprocess.run(['cp', DRIVE_ZIP, ZIP], check=True)
elif not os.path.exists(ZIP):
    print('Downloading D-Fire from the authors OneDrive (about 3 GB) ...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', ZIP, onedrive_direct(ONEDRIVE)])
assert os.path.exists(ZIP) and zipfile.is_zipfile(ZIP), (
    'Download failed. Upload D-Fire.zip to Google Drive folder MyDrive/HazardWatch/ and run this cell again.')
print('Unzipping ...'); zipfile.ZipFile(ZIP).extractall(DATA)

# Find the train / test image folders whatever the zip layout is.
img_dirs = sorted({os.path.dirname(p) for p in glob.glob(f'{DATA}/**/images/*.jpg', recursive=True)})
print(img_dirs)
train = next(d for d in img_dirs if 'train' in d.lower())
val = next(d for d in img_dirs if 'test' in d.lower() or 'val' in d.lower())
for d in (train, val):
    n_img = len(glob.glob(d + '/*.jpg')); n_lab = len(glob.glob(d.replace('images', 'labels') + '/*.txt'))
    print(d, n_img, 'images', n_lab, 'labels')


In [ ]:
# 3. Check the class ids on the labels (D-Fire: 0 = smoke, 1 = fire).
import collections
counts = collections.Counter()
for f in glob.glob(train.replace('images', 'labels') + '/*.txt'):
    for line in open(f):
        if line.strip(): counts[line.split()[0]] += 1
print('boxes per class id:', dict(counts))
assert set(counts) <= {'0', '1'}, 'unexpected class ids'
yaml = f'''path: /
train: {train}
val: {val}
names:
  0: smoke
  1: fire
'''
open('/content/dfire.yaml', 'w').write(yaml); print(yaml)

In [ ]:
# 4. Train YOLO11n (nano: fast enough for a CPU or phone). Resumes automatically after a disconnect.
from ultralytics import YOLO
last = f'{OUT}/train/weights/last.pt'
if os.path.exists(last):
    print('Resuming from', last)
    model = YOLO(last); model.train(resume=True)
else:
    model = YOLO('yolo11n.pt')
    model.train(data='/content/dfire.yaml', epochs=40, imgsz=640, batch=32, workers=2,
                project=OUT, name='train', exist_ok=True, patience=10, save_period=1, seed=0)

In [ ]:
# 5. Evaluate on the D-Fire test set.
best = f'{OUT}/train/weights/best.pt'
m = YOLO(best)
r = m.val(data='/content/dfire.yaml', imgsz=640, batch=32)
import json
summary = {'mAP50': float(r.box.map50), 'mAP50_95': float(r.box.map),
           'per_class_mAP50': {m.names[i]: float(v) for i, v in enumerate(r.box.ap50)},
           'precision': float(r.box.mp), 'recall': float(r.box.mr)}
print(json.dumps(summary, indent=2)); open(f'{OUT}/results.json', 'w').write(json.dumps(summary, indent=2))

In [ ]:
# 6. Export for the PC (ONNX, CPU) and keep everything in Drive.
onnx = m.export(format='onnx', imgsz=640, simplify=True)
!cp {onnx} {OUT}/
print('Done. Files in Google Drive:', OUT); print(os.listdir(OUT))